In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/fraud-detection-mlops

Mounted at /content/drive
/content/drive/MyDrive/fraud-detection-mlops


In [4]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


df = pd.read_csv("data/raw/creditcard.csv")
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape)
print(X_test.shape)

# sklearn Pipeline
pipeline = Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])

# Modell trainieren
pipeline.fit(X_train, y_train)
print("Training abgeschlossen.")

# Vorhersage erzeugen
y_pred = pipeline.predict(X_test)

# Classification Report
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ROC-AUC berechnen
y_prob = pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {roc_auc}")

# Metriken speichern
results = {"roc_auc": float(roc_auc)}
with open("reports/model_metrics.json", "w") as f:
    json.dump(results, f, indent=4)





(227845, 30)
(56962, 30)
Training abgeschlossen.
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.94      0.82      0.87        98

    accuracy                           1.00     56962
   macro avg       0.97      0.91      0.94     56962
weighted avg       1.00      1.00      1.00     56962

[[56859     5]
 [   18    80]]
ROC-AUC: 0.9630272515590367


In [6]:
import joblib
joblib.dump(pipeline, "models/random_forest_pipeline.pkl")
print("Pipeline gespeichert.")

Pipeline gespeichert.


**Pipeline testen**

Modell testen

In [8]:
loaded_pipeline = joblib.load("models/random_forest_pipeline.pkl")
print("Pipeline geladen.")

Pipeline geladen.


In [11]:
import numpy as np
loaded_pred = loaded_pipeline.predict(X_test)
print(loaded_pred[:10])

# Vergleich
print(np.array_equal(y_pred, loaded_pred))

[0 0 0 0 0 0 0 0 0 0]
True


In [12]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_data_ingestion.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/02_preprocessing.ipynb
	reports/model_metrics.json

no changes added to commit (use "git add" and/or "git commit -a")


In [13]:
!cat reports/model_metrics.json

{
    "roc_auc": 0.9630272515590367
}

Datei wurde korrekt geschrieben

In [14]:
!ls -lh models

total 2.6M
-rw------- 1 root root 2.6M Aug 21 10:26 random_forest_pipeline.pkl


In [19]:
!git add .
!git commit -m "Add preprocessing pipeline, evaluation and model artifact"
!git push

[main a32ae46] Add preprocessing pipeline, evaluation and model artifact
 1 file changed, 1 insertion(+), 1 deletion(-)
To https://github.com/5b0chmann/fraud-detection-mlops.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/5b0chmann/fraud-detection-mlops.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.
